# 01 · Looking at the ACDC data before any model touches it

This notebook only reads the **100 official training patients**. The 50 test patients are not opened
here, so nothing I see can influence choices that are later judged on the test set.

It saves every figure to `figures/` (the README and `docs/` use them) instead of showing them inline,
which keeps the notebook small enough to commit.

In [1]:
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from cardiac.data.io import ACDC_GROUPS, LABELS, load_frame, load_metadata
from cardiac.plots import middle_slice, save, show_slice, structure_legend

ROOT = Path("../data/raw/acdc")
FIGURES = Path("../figures")
metas = load_metadata(ROOT, "train")
print(len(metas), "training patients:", dict(Counter(m.group for m in metas)))

100 training patients: {'DCM': 20, 'HCM': 20, 'MINF': 20, 'NOR': 20, 'RV': 20}


## 1. One heart from each group

Each row is one patient at end-diastole (ED, heart fully filled) and end-systole (ES, fully squeezed),
on the slice with the most LV cavity at ED. Overlays are the expert labels.

Look at the HCM row: the ES image has no labels at all. The heart's base moves towards the apex as it
contracts, so a slice near the base at ED can be above the ventricles at ES. Same slice index does
not mean same piece of heart, which is one reason every measurement is computed on whole 3D stacks.

In [2]:
first_of_group = {g: next(m for m in metas if m.group == g) for g in ACDC_GROUPS}
fig, axes = plt.subplots(len(ACDC_GROUPS), 2, figsize=(5, 12))
for row, (group, meta) in enumerate(first_of_group.items()):
    ed, es = load_frame(ROOT, meta, "ed"), load_frame(ROOT, meta, "es")
    k = middle_slice(ed.label)
    show_slice(axes[row, 0], ed.image[:, :, k], ed.label[:, :, k])
    show_slice(axes[row, 1], es.image[:, :, k], es.label[:, :, k])
    axes[row, 0].set_title(f"{group} · {meta.pid} · ED", fontsize=9)
    axes[row, 1].set_title("ES", fontsize=9)
fig.legend(handles=structure_legend(), loc="lower center", ncol=3, fontsize=8, frameon=False)
save(fig, FIGURES / "data_groups_ed_es.png")

## 2. A whole stack, base to apex

The short axis cuts the heart like a loaf of bread. Near the base (top of the heart) the RV is large
and the outlines are messy because of the valves; near the apex everything shrinks. These end slices
are where segmentation is hardest.

In [3]:
meta = first_of_group["NOR"]
ed = load_frame(ROOT, meta, "ed")
n = ed.image.shape[2]
cols = 5
fig, axes = plt.subplots(int(np.ceil(n / cols)), cols, figsize=(10, 2.2 * np.ceil(n / cols)))
for k, ax in enumerate(axes.flat):
    if k < n:
        show_slice(ax, ed.image[:, :, k], ed.label[:, :, k])
        ax.set_title(f"slice {k}", fontsize=8)
    else:
        ax.set_axis_off()
fig.suptitle(f"{meta.pid} (NOR), all ED slices", fontsize=10)
save(fig, FIGURES / "data_slice_stack.png")

## 3. Spacing: why millimetres matter

The mirror resampled every scan to 1 × 1 × 10 mm. The *original* scans were not uniform: in-plane pixels
ranged from about 0.7 to 1.9 mm and slice gaps from 5 to 10 mm. The area of one pixel alone differs by
up to 7× between scanners, so counting pixels without their spacing would give meaningless volumes.

In [4]:
original = pd.DataFrame([m.original_spacing_mm for m in metas], columns=["x_mm", "y_mm", "z_mm"])
fig, axes = plt.subplots(1, 2, figsize=(8, 3))
axes[0].hist(original.x_mm, bins=15, color="#0E7C86")
axes[0].set(xlabel="original in-plane pixel size (mm)", ylabel="patients")
axes[1].hist(original.z_mm, bins=[4.5, 5.5, 6.5, 7.5, 8.5, 9.5, 10.5], color="#0E7C86")
axes[1].set(xlabel="original slice gap (mm)")
fig.suptitle(
    "Original ACDC spacing (training patients); the mirror resampled all to 1 x 1 x 10 mm",
    fontsize=9,
)
save(fig, FIGURES / "data_original_spacing.png")
print(original.describe().loc[["min", "50%", "max"]].round(2))

     x_mm  y_mm  z_mm
min  0.70  0.70   5.0
50%  1.56  1.56  10.0
max  1.92  1.92  10.0


## 4. Class balance

Background dominates. This is why a plain pixel-accuracy score is useless here (predicting "background"
everywhere would already be about 88% accurate, even on these heart-centred crops) and why the
loss includes a Dice term.

In [5]:
counts = Counter()
for meta in metas:
    for phase in ("ed", "es"):
        values, n = np.unique(load_frame(ROOT, meta, phase).label, return_counts=True)
        counts.update(dict(zip(values.tolist(), n.tolist(), strict=True)))
total = sum(counts.values())
share = {LABELS[k]: counts[k] / total * 100 for k in sorted(counts)}
balance = pd.Series(share, name="% of voxels")
print(balance.round(2))
fig, ax = plt.subplots(figsize=(5, 2.6))
ax.bar(balance.index, balance.values, color=["#9CA3AF", "#3B82F6", "#22C55E", "#EF4444"])
ax.set(ylabel="% of all voxels", yscale="log", title="Voxel share per label (log scale)")
save(fig, FIGURES / "data_class_balance.png")

background    87.90
RV             3.76
MYO            4.18
LV             4.16
Name: % of voxels, dtype: float64


## 5. The diseases are visible in simple numbers

LV volume at ED and LV ejection fraction, computed straight from the expert masks with the mirror's
spacing (voxel volume = 1 × 1 × 10 mm³ = 0.01 mL). DCM hearts are large and pump weakly; HCM hearts pump
strongly. This is the signal the Phase 5 classifier will use. These numbers are from the training set
and the expert labels only; nothing here is a model result.

In [6]:
rows = []
for meta in metas:
    ed, es = load_frame(ROOT, meta, "ed"), load_frame(ROOT, meta, "es")
    voxel_ml = np.prod(ed.spacing_mm) / 1000.0
    edv = (ed.label == 3).sum() * voxel_ml
    esv = (es.label == 3).sum() * voxel_ml
    rows.append({"group": meta.group, "lv_edv_ml": edv, "lv_ef_pct": (edv - esv) / edv * 100})
volumes = pd.DataFrame(rows)
print(volumes.groupby("group")[["lv_edv_ml", "lv_ef_pct"]].median().loc[list(ACDC_GROUPS)].round(1))

fig, axes = plt.subplots(1, 2, figsize=(9, 3.2))
for ax, column, label in (
    (axes[0], "lv_edv_ml", "LV end-diastolic volume (mL)"),
    (axes[1], "lv_ef_pct", "LV ejection fraction (%)"),
):
    data = [volumes.loc[volumes.group == g, column] for g in ACDC_GROUPS]
    ax.boxplot(data, tick_labels=ACDC_GROUPS)
    ax.set(ylabel=label)
fig.suptitle("Expert-label measurements, 100 training patients", fontsize=9)
save(fig, FIGURES / "data_lv_by_group.png")

       lv_edv_ml  lv_ef_pct
group                      
NOR        135.0       60.0
MINF       165.6       32.7
DCM        287.6       16.0
HCM        142.5       68.3
RV         101.7       56.0
